# ¿Qué patrones temporales explican cómo una canción entra, crece y desaparece del chart en Argentina, y qué tan predecible es ese comportamiento?

## 02 · Resumen general

Este notebook es el **resumen para las slides / el video** (10 min). No repite el
análisis completo: trae, de cada sub-pregunta, **el gráfico más fuerte y la conclusión**,
en el orden 03 → 04 → 05. Se puede leer solo, sin abrir los otros notebooks.

Tratamos la evolución diaria de streams / posición en el Top 200 de Spotify como una
**señal temporal** y le aplicamos las herramientas de la materia (Fourier, filtrado,
energía, autocorrelación, correlación cruzada, estacionariedad, forecasting). Las tres
sub-preguntas:

1. **Forma de la señal** — ¿cómo es el ciclo de vida de un hit? (notebook 03)
2. **Comparación entre señales** — ¿Argentina sigue al mundo? (notebook 04)
3. **Predicción** — ¿qué tan predecible es la serie? (notebook 05)

## Setup

In [ ]:
import sys, logging
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import construir_serie, interpolar_serie, ajustar_arima, ajustar_prophet
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
logging.getLogger("prophet").setLevel(logging.ERROR)

df_ar = pd.read_csv("../data/charts_argentina.csv", parse_dates=["date"])

def serie_cancion(cancion):
    cruda = construir_serie(df_ar, {"title": cancion, "region": "Argentina", "chart": "top200",
                                    "valor": "streams", "agregado": "sum"})
    return interpolar_serie(cruda)

## 1. Ciclo de vida: viral vs. catálogo (sub-pregunta 1 — notebook 03)

**Gráfico más fuerte:** la forma de onda de un hit **viral** contra la de un tema de
**catálogo**, alineadas por "días desde la entrada al chart".

In [ ]:
serie_viral = serie_cancion("C.R.O: Bzrp Music Sessions, Vol. 29")
serie_cat   = serie_cancion("Pasos Al Costado")

dias_v = (serie_viral.index - serie_viral.index[0]).days
dias_c = (serie_cat.index - serie_cat.index[0]).days

plt.figure(figsize=(12, 5))
plt.plot(dias_v, serie_viral.values, color="crimson", label="Viral — C.R.O: Bzrp Vol. 29")
plt.plot(dias_c, serie_cat.values, color="steelblue", label="Catálogo — Pasos Al Costado (Turf)")
plt.title("Ciclo de vida: hit viral vs. tema de catálogo")
plt.xlabel("Días desde la entrada al chart"); plt.ylabel("streams diarios"); plt.legend()
plt.tight_layout(); plt.show()

**Conclusión.** El hit viral es un **transitorio no estacionario**: entra fuerte,
pega el pico en ~3 días (pendiente de subida ~63× la de caída) y decae de forma sostenida
durante meses; en frecuencia, energía concentrada en baja frecuencia, y una
autocorrelación que decae lento. El tema de catálogo es casi una **meseta cuasi-
estacionaria**: entra sin salto, se sostiene años en una banda angosta de streams y deja
ver mejor la modulación **semanal**. En términos de la materia: la viral es un **evento**,
el catálogo es un **régimen**.

## 2. Argentina vs. Global (sub-pregunta 2 — notebook 04)

**Gráfico más fuerte:** cantidad de artistas en común por día entre el Top 200 de
Argentina y el Global (intersección real de conjuntos).

In [ ]:
df_ag = pd.read_csv("../data/charts_ar_global.csv", parse_dates=["date"])
top200 = df_ag[df_ag["chart"] == "top200"].copy()
top200["ai"] = top200["artist"].str.split(", ")
ex = top200.explode("ai")
set_ar = ex[ex["region"] == "Argentina"].groupby("date")["ai"].apply(set)
set_gl = ex[ex["region"] == "Global"].groupby("date")["ai"].apply(set)
fechas = set_ar.index.intersection(set_gl.index)
inter = pd.Series({d: len(set_ar[d] & set_gl[d]) for d in fechas}).sort_index()
inter_semanal = inter.resample("W").mean()

plt.figure(figsize=(12, 5))
plt.plot(inter.index, inter.values, color="lightgray", lw=0.8, label="Diaria")
plt.plot(inter_semanal.index, inter_semanal.values, color="darkblue", lw=1.6, label="Promedio semanal")
plt.title("Artistas en común por día — Top 200 Argentina ∩ Global")
plt.xlabel("Fecha"); plt.ylabel("artistas en común"); plt.legend()
plt.tight_layout(); plt.show()
print(f"Media diaria: {inter.mean():.0f} artistas en común (desvío {inter.std():.0f})")

**Conclusión.** El solapamiento es **alto** (~65 artistas en común por día): buena
parte del ranking argentino está poblado por los mismos artistas que dominan el mundo, con
un pico histórico en **2017** (crossover latino: Despacito, Shape of You, CNCO…). La
**correlación cruzada** de la actividad AR vs. Global es débil (~0.24) y **casi simétrica
alrededor de lag 0** (máximo en lag 28 pero empatado con lag 0): **no hay desfasaje
temporal claro** — Argentina no lidera ni va a la zaga, **acompaña al Global casi en
tiempo real**, conservando un margen local (cumbia, rock nacional, trap) que le da
identidad parcial.

## 3. Predicción: ¿qué tan predecible es? (sub-pregunta 3 — notebook 05)

**Gráfico más fuerte:** el MAE de cada modelo contra el **baseline naive** (repetir el
último valor), en los dos backtests. La barra de referencia es el naive.

In [ ]:
H = 25
orden = (1, 1, 1)   # elegido por AIC en el notebook 05

def mae(pred, real):
    return float(np.mean(np.abs(np.asarray(pred) - real.values)))

def backtest(train, test):
    a = ajustar_arima(train, orden)["modelo"]
    pa = np.asarray(a.forecast(steps=len(test)))
    pn = np.repeat(train.values[-1], len(test))
    res = {"Naive": mae(pn, test), "ARIMA(1,1,1)": mae(pa, test)}
    try:  # Prophet puede fallar si el backend de Stan no carga en el entorno
        mp = ajustar_prophet(train)
        pp = mp.predict(mp.make_future_dataframe(periods=len(test), freq="D"))["yhat"].to_numpy()[-len(test):]
        res["Prophet"] = mae(pp, test)
    except Exception as e:
        print(f"  [aviso] Prophet no disponible ({type(e).__name__}); se omite en este backtest.")
        res["Prophet"] = float("nan")
    return res

# A: meseta final | B: caída post-pico (entrena primeros 20 días)
res_A = backtest(serie_viral.iloc[:-H], serie_viral.iloc[-H:])
res_B = backtest(serie_viral.iloc[:20], serie_viral.iloc[20:20 + H])

modelos = ["Naive", "ARIMA(1,1,1)", "Prophet"]
x = np.arange(len(modelos)); w = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, [res_A[m] for m in modelos], w, label="A · meseta final", color="#8ecae6")
ax.bar(x + w/2, [res_B[m] for m in modelos], w, label="B · caída post-pico", color="#fb8500")
ax.set_yscale("log")
ax.set_xticks(x); ax.set_xticklabels(modelos)
ax.set_ylabel("MAE (streams, escala log)")
ax.set_title("Error de pronóstico vs. baseline naive — dos backtests")
ax.legend()
for i, m in enumerate(modelos):
    if not np.isnan(res_A[m]):
        ax.annotate(f"{res_A[m]:,.0f}", (i - w/2, res_A[m]), ha="center", va="bottom", fontsize=8)
    if not np.isnan(res_B[m]):
        ax.annotate(f"{res_B[m]:,.0f}", (i + w/2, res_B[m]), ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()

print("Backtest A (meseta):", {k: round(v) for k, v in res_A.items()})
print("Backtest B (caída) :", {k: round(v) for k, v in res_B.items()})

**Conclusión — y esto es parte de la respuesta a "qué tan predecible es".** Se
corrieron **dos backtests**: uno sobre la **meseta plana** final y otro sobre la **caída
empinada post-pico**. **En ningún tramo los modelos superaron al baseline naive.** En la
meseta, ARIMA(1,1,1) **empata** al naive (MAE ~2.400) simplemente porque la serie ya está
plana; en la caída, donde sí hay dinámica real, ARIMA **vuelve a empatar** al naive
(~16.750): al ser un random-walk **sin término de deriva**, pronostica un valor constante
y arrastra la bajada igual que la persistencia. **Prophet además empeora bastante** en
ambos tramos (MAE ~106k y ~512k): con pocos datos y series cortas sobre-extrapola.

**Conclusión honesta:** esta serie viral **no fue bien capturada por ARIMA(1,1,1) ni por
Prophet** — ninguno le gana al naive. Para superar a la persistencia haría falta un modelo
con **término de deriva explícito** (ARIMA con tendencia, o modelar el decaimiento
exponencial / log-streams). El ciclo de vida es **estructuralmente legible** (subida-pico-
caída) pero, a nivel de valor diario puntual, **poco predecible más allá de la
persistencia** con las herramientas estándar probadas.

## Cierre

**Respuesta corta a la pregunta central.** Una canción en el chart argentino sigue un
patrón temporal claro —**entrada abrupta → pico → caída** (viral) o **meseta sostenida**
(catálogo)—, con una **estacionalidad semanal** de fondo, y su consumo está **fuertemente
alineado con las tendencias globales, casi sin desfasaje**. Ese patrón es **descriptible y
comparable** con las herramientas de la materia; pero en cuanto a **predicción puntual**,
la serie viral resultó **poco predecible más allá del baseline naive**: ni ARIMA(1,1,1) ni
Prophet lo superaron, lo que marca el límite del enfoque y el camino a seguir (modelos con
deriva).